# 01 – Analyse exploratoire des sources (EDA)

Exploration des données brutes collectées depuis les 4 sources médicales.

**Sources** : MediQA · FrenchMedMCQA · MedQuAD · UltraMedical

In [ ]:
import json
import sys
from pathlib import Path
from collections import Counter

sys.path.insert(0, str(Path('..')))
RAW_DIR = Path('../data/raw')

## 1. Chargement des sources brutes

In [ ]:
def load_jsonl(path):
    records = []
    with open(path, encoding='utf-8') as f:
        for line in f:
            records.append(json.loads(line))
    return records

sources = {
    'mediqa':        load_jsonl(RAW_DIR / 'mediqa/mediqa_raw.jsonl'),
    'frenchmedmcqa': load_jsonl(RAW_DIR / 'frenchmedmcqa/frenchmedmcqa_raw.jsonl'),
    'medquad':       load_jsonl(RAW_DIR / 'medquad/medquad_raw.jsonl'),
    'ultramedical':  load_jsonl(RAW_DIR / 'ultramedical_preference/ultramedical_sft_raw.jsonl'),
    'ultramedical_dpo': load_jsonl(RAW_DIR / 'ultramedical_preference/ultramedical_dpo_raw.jsonl'),
}

for name, records in sources.items():
    print(f'{name:25s} : {len(records):>5} enregistrements')

## 2. Distribution des langues

In [ ]:
lang_counts = Counter()
for name, records in sources.items():
    for r in records:
        lang_counts[r.get('language', 'unknown')] += 1

total = sum(lang_counts.values())
print('Distribution des langues (SFT + DPO brut) :')
for lang, count in lang_counts.most_common():
    print(f'  {lang:4s} : {count:>5} ({count/total*100:.1f}%)')

## 3. Longueur des questions et réponses

In [ ]:
def word_count(text):
    return len(str(text).split())

print(f'{'Source':<25} {'Q min':>6} {'Q moy':>6} {'Q max':>6} {'R min':>6} {'R moy':>6} {'R max':>6}')
print('-' * 70)

for name, records in sources.items():
    if name == 'ultramedical_dpo':
        q_lens = [word_count(r.get('prompt', '')) for r in records]
        r_lens = [word_count(r.get('chosen', '')) for r in records]
    else:
        q_lens = [word_count(r.get('question', '')) for r in records]
        r_lens = [word_count(r.get('answer', '')) for r in records]

    print(f'{name:<25} {min(q_lens):>6} {sum(q_lens)//len(q_lens):>6} {max(q_lens):>6} '
          f'{min(r_lens):>6} {sum(r_lens)//len(r_lens):>6} {max(r_lens):>6}')

## 4. Exemples aléatoires par source

In [ ]:
import random
random.seed(42)

for name, records in list(sources.items())[:4]:
    sample = random.choice(records)
    print(f'\n{'='*60}')
    print(f'SOURCE : {name.upper()} ({sample.get("language", "?")})')
    print(f'{'='*60}')
    q = sample.get('question', sample.get('prompt', ''))[:300]
    a = sample.get('answer', sample.get('chosen', ''))[:300]
    print(f'Q: {q}...' if len(q) == 300 else f'Q: {q}')
    print(f'A: {a}...' if len(a) == 300 else f'A: {a}')

## 5. Détection de doublons potentiels

In [ ]:
all_questions = []
for name, records in sources.items():
    for r in records:
        q = r.get('question', r.get('prompt', '')).strip().lower()[:100]
        all_questions.append(q)

question_counts = Counter(all_questions)
duplicates = {q: c for q, c in question_counts.items() if c > 1}
print(f'Total questions : {len(all_questions)}')
print(f'Doublons exacts (premiers 100 chars) : {len(duplicates)}')
if duplicates:
    print('\nExemples :')
    for q, c in list(duplicates.items())[:3]:
        print(f'  [{c}x] "{q[:80]}..."')

## 6. Résumé — Décision sur les volumes

| Source | Langue | Brut | Retenu (après filtre) |
|---|---|---|---|
| MediQA | EN | 1 000 | ~948 (filtre longueur max) |
| FrenchMedMCQA | FR | 1 500 | 1 500 (réponse reconstruite) |
| MedQuAD | EN | 2 000 | 2 000 |
| UltraMedical (SFT) | EN | 500 | 500 |
| UltraMedical (DPO) | EN | 2 000 | 2 000 |

**Remarques :**
- FrenchMedMCQA : réponses QCM reconstruites au format `"La réponse correcte est X : [texte option]"`
- MediQA : ~52 cas trop longs (>2000 mots) exclus
- DPO filtré sur écart de score ≥ 0.3 entre chosen et rejected